# AutoMail — AI Video Clips (Kaggle Free T4 GPU)

**Pipeline:** Text prompt → Flux AI Image → Stable Video Diffusion → MP4 clip

> **Before running:** Settings → Accelerator → **GPU T4 x1** → Save

Uses `stabilityai/stable-video-diffusion-img2vid-xt` — battle-tested on T4, zero version conflicts.

In [ ]:
# CELL 1 — Verify GPU
import torch
assert torch.cuda.is_available(), '❌ No GPU! Go to Settings → Accelerator → GPU T4 x1 → Save'
print(f'✅ GPU : {torch.cuda.get_device_name(0)}')
print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

In [ ]:
# CELL 2 — Install (SVD uses standard diffusers, no special versions needed)
!pip install -q diffusers transformers accelerate
!pip install -q imageio imageio-ffmpeg Pillow requests
print('✅ Installed')

In [ ]:
# CELL 3 — Load Stable Video Diffusion pipeline
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video

print('Loading SVD pipeline...')
pipe = StableVideoDiffusionPipeline.from_pretrained(
    'stabilityai/stable-video-diffusion-img2vid-xt',
    torch_dtype=torch.float16,
    variant='fp16',
)
pipe.enable_model_cpu_offload()
pipe.unet.enable_forward_chunking()
print('✅ SVD loaded')

In [ ]:
# CELL 4 — Generate cinematic images via Pollinations Flux (free, no key needed)
import requests, time
from pathlib import Path
from PIL import Image
from io import BytesIO

IMG_DIR = Path('/kaggle/working/automail_images')
IMG_DIR.mkdir(exist_ok=True)

SCENES = [
    {
        'scene': 1, 'label': 'HOOK',
        'image_prompt': (
            'cinematic close up tired young businesswoman at cluttered desk late at night, '
            'blue laptop screen glow on stressed face, dark moody room, film still, '
            'shallow depth of field, professional photography, 4K'
        ),
        'motion_frames': 25,
        'motion_bucket': 100,
    },
    {
        'scene': 2, 'label': 'SOLUTION',
        'image_prompt': (
            'cinematic medium shot confident smiling entrepreneur at modern glass desk, '
            'laptop with colorful email dashboard, warm golden sunlight large windows, '
            'shallow depth of field, professional commercial photography, 4K'
        ),
        'motion_frames': 25,
        'motion_bucket': 110,
    },
    {
        'scene': 3, 'label': 'BENEFITS',
        'image_prompt': (
            'cinematic wide shot three diverse business professionals celebrating '
            'in bright modern office, city skyline through floor-to-ceiling windows, '
            'golden hour sunset light, high energy, professional photography, 4K'
        ),
        'motion_frames': 25,
        'motion_bucket': 127,
    },
    {
        'scene': 4, 'label': 'CTA',
        'image_prompt': (
            'cinematic premium product shot sleek laptop on white marble desk, '
            'dark studio background dramatic spotlight, glowing screen orange dashboard, '
            'luxury advertisement, professional photography, 4K'
        ),
        'motion_frames': 25,
        'motion_bucket': 90,
    },
]

print('Generating 4 cinematic images via Pollinations Flux...')
for sc in SCENES:
    n   = sc['scene']
    out = IMG_DIR / f'scene_{n}_{sc["label"].lower()}.png'
    if out.exists():
        print(f'  [SKIP] Scene {n} image cached')
        continue
    url = ('https://image.pollinations.ai/prompt/' +
           requests.utils.quote(sc['image_prompt']) +
           '?width=1024&height=576&seed=' + str(n*100) +
           '&model=flux&nologo=true')
    print(f'  Fetching Scene {n} {sc["label"]}...')
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=60)
            img = Image.open(BytesIO(r.content)).convert('RGB')
            img.save(out)
            print(f'  ✅ Scene {n}: {img.size}  → {out.name}')
            break
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(5)

print('✅ All images generated')

In [ ]:
# CELL 5 — Animate each image into a video clip with SVD
import torch
from diffusers.utils import load_image, export_to_video
from pathlib import Path
import time

CLIP_DIR = Path('/kaggle/working/automail_clips')
CLIP_DIR.mkdir(exist_ok=True)
generated = []

for sc in SCENES:
    n     = sc['scene']
    label = sc['label']
    img_p = Path(f'/kaggle/working/automail_images/scene_{n}_{label.lower()}.png')
    out   = CLIP_DIR / f'scene_{n}_{label.lower()}.mp4'

    if out.exists():
        print(f'[Scene {n}] {label} — cached')
        generated.append(str(out))
        continue

    print(f'\n[Scene {n}] {label} — animating with SVD...')
    t0 = time.time()

    image = load_image(str(img_p))
    image = image.resize((1024, 576))  # SVD XT native resolution

    with torch.no_grad():
        frames = pipe(
            image,
            num_frames=sc['motion_frames'],
            motion_bucket_id=sc['motion_bucket'],
            fps=7,
            decode_chunk_size=4,
            generator=torch.manual_seed(n * 42),
        ).frames[0]

    export_to_video(frames, str(out), fps=7)
    elapsed = time.time() - t0
    mb = out.stat().st_size / 1_048_576
    print(f'✅ Scene {n} done — {elapsed:.0f}s | {mb:.1f} MB → {out.name}')
    generated.append(str(out))
    torch.cuda.empty_cache()

print(f'\n✅ All {len(generated)} clips done!')

In [ ]:
# CELL 6 — Verify & zip for download
import zipfile
from pathlib import Path

clips = sorted(Path('/kaggle/working/automail_clips').glob('*.mp4'))
print('Generated clips:')
for c in clips:
    print(f'  ✅ {c.name}  ({c.stat().st_size/1_048_576:.1f} MB)')

ZIP = Path('/kaggle/working/automail_clips.zip')
with zipfile.ZipFile(ZIP, 'w') as zf:
    for c in clips:
        zf.write(c, c.name)

print(f'\n✅ ZIP: {ZIP.stat().st_size/1_048_576:.1f} MB')
print('\n📥 Download → Kaggle right panel → Output → automail_clips.zip')
print('   Extract 4 mp4 files → paste into:  output/wan21_raw/')
print('   Then run:  python automail_wan21_local.py')

In [ ]:
# CELL 7 — Preview
from IPython.display import Video, display
from pathlib import Path

for f in sorted(Path('/kaggle/working/automail_clips').glob('*.mp4')):
    print(f'▶ {f.stem}')
    display(Video(str(f), width=700, html_attributes='controls'))